In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 49. Week 33 — MLP, backpropagation, initialization, and Adam

## 学習目標

- affine–tanh–affine graphのforward passを式とcodeで対応付ける
- chain ruleから全parameter gradientを導出する
- centered finite differenceでbackpropを監査する
- initialization、regularization、early stoppingを比較する

## 前提知識

- multivariable calculus、matrix multiplication
- B4のfinite-difference audit
- B9のdevelopment-only fixture contract

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 49


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask

assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
assert set(fixture.partitions) == {"inner_train", "inner_validation"}

print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("numeric / sequence shape:", fixture.numeric_features.shape, fixture.token_hashes.shape)
print("locked outer rows present: False")
print("fixture hash lineage:", fixture.provenance)

fixture rows: 256
inner train / validation: 192 64
numeric / sequence shape: (256, 12) (256, 128)
locked outer rows present: False
fixture hash lineage: {'panel_artifact_sha256': '6c6008c2f28c30299e15e37613cfb0b3b22e8fd283858f5b459227c7e4a412a8', 'previous_filing_sidecar_sha256': '9ff2efef335357ff53bb1e4ba5c57f4b2e8799fc4ee5d830c55843a50026fbbc', 'normalized_manifest_sha256': '1283b9cb0992cfd2caaa942f6c869e212762c90a9abbc9a050173f5e3963daba', 'preanalysis_contract_sha256': 'fbe69fdf3b3bccba7fab70bcbb726d0df61685901cc0322d76fc66be1d7bbd6e'}


In [4]:
numeric_preprocessor = qt.fit_numeric_preprocessor(fixture.numeric_features, train_mask)
numeric_features = numeric_preprocessor.transform(fixture.numeric_features)
numeric_train = numeric_features[train_mask]
numeric_validation = numeric_features[validation_mask]
target_train = fixture.targets[train_mask]
target_validation = fixture.targets[validation_mask]
entity_validation = np.asarray(fixture.entity_ids)[validation_mask]

assert np.all(np.isfinite(numeric_features))
print("processed numeric shape:", numeric_features.shape)

processed numeric shape: (256, 24)


## 1. Computational graph

one-hidden-layer regressorを

$$
Z=XW_1+b_1,\qquad H=\tanh Z,\qquad \hat y=Hw_2+b_2,
$$

$$
L=\frac{1}{2n}\lVert\hat y-y\rVert_2^2
$$

とする。reverse modeでは出力からadjointを逆伝播する。

$$
\bar{\hat y}=\frac{\hat y-y}{n},\quad
\bar w_2=H^\top\bar{\hat y},\quad
\bar Z=(\bar{\hat y}w_2^\top)\odot(1-H^2),\quad
\bar W_1=X^\top\bar Z.
$$

In [5]:
audit_rng = task_rng(1)
audit_features = numeric_train[:6, :3]
audit_target = target_train[:6]
audit_parameters = qt.initialize_mlp(3, 4, rng=audit_rng)
gradient_audit = qt.check_mlp_gradients(
    audit_parameters, audit_features, audit_target, step=1e-6, tolerance=2e-5
)
assert gradient_audit.passed
print("maximum relative gradient error:", gradient_audit.maximum_relative_error)

fig = go.Figure()
fig.add_scatter(
    x=gradient_audit.numerical,
    y=gradient_audit.analytic,
    mode="markers",
    name="parameters",
)
low = float(min(gradient_audit.numerical.min(), gradient_audit.analytic.min()))
high = float(max(gradient_audit.numerical.max(), gradient_audit.analytic.max()))
fig.add_scatter(x=[low, high], y=[low, high], mode="lines", name="identity")
fig.update_layout(
    title="Backpropagation audit",
    xaxis_title="Centered finite difference",
    yaxis_title="Analytic gradient",
    template="plotly_white",
)
fig.show()

maximum relative gradient error: 2.2674717174733067e-10


## 2. Initializationとtraining trace

Xavier scaleはfan-in/fan-outに応じてactivation varianceの崩壊・爆発を抑える。Adamはgradientの一次・二次momentを追跡するが、optimizer名だけで収束や一般化は保証しない。validationはparameter更新に使わずearly-stopping epochの選択だけに使う。

In [6]:
mlp_result = qt.train_mlp(
    numeric_train,
    target_train,
    numeric_validation,
    target_validation,
    hidden_width=16,
    learning_rate=0.003,
    epochs=200,
    patience=20,
    l2=1e-4,
    rng=task_rng(2),
)
mlp_validation = qt.mlp_predict(mlp_result.parameters, numeric_validation)
zero_validation = np.zeros_like(target_validation)
metrics = pd.DataFrame(
    [
        {"model": "zero", **qt.regression_error_table(target_validation, zero_validation, entity_validation)},
        {"model": "numeric_mlp", **qt.regression_error_table(target_validation, mlp_validation, entity_validation)},
    ]
)
display(metrics)

fig = go.Figure()
fig.add_scatter(y=mlp_result.training_losses, mode="lines", name="training loss")
fig.add_scatter(y=mlp_result.validation_losses, mode="lines", name="validation loss")
fig.add_vline(x=mlp_result.best_epoch, line_dash="dash")
fig.update_layout(
    title="Full-batch Adam trace",
    xaxis_title="Epoch",
    yaxis_title="Half MSE",
    yaxis_type="log",
    template="plotly_white",
)
fig.show()

,model,mae,median_absolute_error,rmse,company_macro_mae
0,zero,0.049469,0.020475,0.114476,0.043651
1,numeric_mlp,0.130387,0.090153,0.211169,0.117417


## 3. 失敗モード

- gradient checkを1 parameterだけで済ませる
- objective scaleが小さいだけでconvergedと判定する
- validationをgradient updateへ混ぜる
- best epochをouter testから選ぶ
- initialization seedを変えた1回の改善をarchitecture効果と呼ぶ

## 4. 段階別演習

### 基礎

1. output biasのgradientを導出せよ。
2. tanh derivativeをcodeと式で照合せよ。

### 標準

3. ReLUへ変更しdead-unit割合を記録せよ。
4. hidden width 16/32、seed 3本を同じ200 epoch上限で比較せよ。

### 研究

5. Adamとfull-batch gradient descentをruntime、best epoch、validation MAEで比較せよ。

## 5. Exit Criteria

- [ ] forward graphとreverse graphを対応付けた
- [ ] 全parameterのcentered finite-difference auditを通した
- [ ] initialization、seed、regularizationを記録した
- [ ] trainingとvalidation traceを分けた
- [ ] fixture結果をcandidate nominationに使っていない

## 6. 出典


- [Goodfellow, Bengio, and Courville, *Deep Learning*](https://www.deeplearningbook.org/)
- [Glorot and Bengio (2010), Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html)
- [Kingma and Ba (2015), Adam](https://arxiv.org/abs/1412.6980)